# Getting Started with MakeTables

This notebook contains interactive examples from the README to get you started with MakeTables.

## Installation

First, make sure you have MakeTables installed:

```bash
pip install maketables
```

For development installation:
```bash
git clone https://github.com/dsliwka/maketables.git
cd maketables
pip install -e .
```

## Descriptive Statistics Table

Let's start with creating a descriptive statistics table using the classic auto dataset (Note that you can work with any pandas DataFrame but MakeTables also supports loading and saving dta files (Stata) as this facilitates working with variable labels).


In [ ]:
import pandas as pd
import maketables as mt

# Load your data (here using a sample Stata dataset with the import_dta function that also stores variable labels)
df = mt.import_dta("../data/auto.dta")

In [ ]:
# Create descriptive statistics table
mt.DTable(df, vars=["mpg","weight","length"], bycol=["foreign"],
          caption="Descriptive Statistics")



## Regression Tables

### Using PyFixest
Use the versatile [PyFixest](https://py-econometrics.github.io/pyfixest/pyfixest.html) package.

In [ ]:
import pyfixest as pf

# Fit your models here using pyfixest
est1 = pf.feols("mpg ~ weight", data=df)
est2 = pf.feols("mpg ~ weight * length", data=df)
est3 = pf.feols("mpg ~ weight * length | foreign", data=df)

# Make the table
mt.ETable([est1, est2, est3], caption = "Regression Results",)



### Using Statsmodels

MakeTables also works with [Statsmodels](https://www.statsmodels.org/stable/index.html) for various types of regression models. Here is for instance a linear probability model and a probit.

In [ ]:
import statsmodels.formula.api as smf

# Generate a dummy variable and label it
df["foreign_i"] = (df["foreign"] == "Foreign")*1
mt.set_var_labels(df, {"foreign_i": "Foreign (indicator)"})

# Fit your models 
est1 = smf.ols("foreign_i ~ weight + length + price", data=df).fit()
est2 = smf.probit("foreign_i ~ weight + length + price", data=df).fit(disp=0)

# Make the table
mt.ETable([est1, est2],  
          model_heads=["OLS","Probit"],
          caption="Regression Results (Statsmodels)")



### Combining different packages

You can also combine models from different packages in one table. 

In [ ]:
# Here we just reestimaate the linear probability model using pyfixest instead of statsmodels
est0=pf.feols("foreign_i ~ weight + length + price", data=df)

# Make the table
mt.ETable([est0, est1, est2],  
          model_heads=["OLS (PyFixest)","OLS (Statsmodels)","Probit"],
          caption="Regression Results combining different packages")
